In [2]:
!pip install python-dotenv

In [3]:
!pip install boto3 pyarrow pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.4/84.4 kB 5.7 MB/s eta 0:00:00


In [4]:
!pip install farm-haystack

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.9/153.9 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 764.0/764.0 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.7/48.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.8/77.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 8.3 MB/s eta 0:00:00
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sha256=532de12db8dd18b2355fc863439df38159dc24d7db4bf1fbc4185af94caaf487
  Stored in directory: /root/.cache/pip/wheels/1a/b0/8c/4b75c4116c31f83c8f9f047231251e13cc74481

In [5]:
import boto3
import pandas as pd
import re
import io
from dotenv import load_dotenv
import os
from haystack.document_stores import InMemoryDocumentStore
from haystack.schema import Document
from haystack.nodes import BM25Retriever, FARMReader
from haystack.pipelines import ExtractiveQAPipeline

# Load environment variables
load_dotenv()

# Fetch credentials from .env
aws_access_key_id = os.getenv("AWS_ACCESS_KEY_ID")
aws_secret_access_key = os.getenv("AWS_SECRET_ACCESS_KEY")
aws_region = os.getenv("AWS_REGION")

# Initialize S3 client using credentials from environment variables
s3_client = boto3.client(
    's3',
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    region_name=aws_region
)

# S3 Bucket and file details
bucket_name = 'content-tagging-lms'
file_key = '4350_vimeo_videos.parquet'

# Fetch the file from S3
response = s3_client.get_object(Bucket=bucket_name, Key=file_key)
parquet_file = response['Body'].read()

# Load only specific columns and limit rows
df = pd.read_parquet(io.BytesIO(parquet_file), columns=['uri', 'transcript_content'])

# Save cleaned transcripts and upload to S3
def clean_transcript(transcript):
    if not transcript:
        return None
    cleaned = re.sub(r'WEBVTT\n\n', '', transcript)
    cleaned = re.sub(r'\d+\n\d{2}:\d{2}:\d{2}\.\d{3} --> \d{2}:\d{2}:\d{2}\.\d{3}\n', '', cleaned)
    cleaned = re.sub(r'\n+', ' ', cleaned).strip()
    return cleaned

df['cleaned_transcript'] = df['transcript_content'].apply(clean_transcript)

# Save cleaned transcript as a Parquet file
cleaned_file_key = '4350_vimeo_videos_cleaned.parquet'
df[['uri', 'cleaned_transcript']].to_parquet(cleaned_file_key, engine='pyarrow')

# Upload the cleaned file to S3
s3_client.upload_file(cleaned_file_key, bucket_name, cleaned_file_key)
print(f"Uploaded cleaned transcript to S3: {cleaned_file_key}")


Uploaded cleaned transcript to S3: 4350_vimeo_videos_cleaned.parquet
